In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic

llm_client = ChatOpenAI(model="gpt-5.4-mini")

result=llm_client.invoke("tell me about npci in 2 words")
result

In [ ]:
result.content

In [ ]:
from langchain.chat_models import init_chat_model

llm_client = init_chat_model(
    model= os.getenv("MODEL_NAME"),
    model_provider= os.getenv("MODEL_PROVIDER"),
    temperature="0.5"
)

llm_client.invoke("Tell me about sharukh khan in 2 words")

In [ ]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage

messages = [
    SystemMessage(content="Give most funnies response"),
    HumanMessage(content="Tell me about sharukh khan in 2 sentences")
]

llm_client.invoke(messages)

In [ ]:
messages = [
    {"role": "system", "content": "Give most funniest response"},
      {"role": "user", "content": "tell me about sharukh khan in 2 sentences"}
]

llm_client.invoke(messages)

In [ ]:
for chunk in llm_client.stream("Tell me about npci in 10 sentences"):
    print(chunk.content, end="")

In [ ]:
messages =[
    "tell me about elon musk in 2 words",
    "tell me about trump in 2 words",
    "tell me about modi in 2 words",

]

llm_client.batch(messages)

In [ ]:
response= llm_client.invoke("tell me about elon musk, his education, his companies, achievements. Gice me response in json format")
response.content

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

parser= JsonOutputParser()
parser.invoke(response)

In [ ]:
chain = llm_client | parser

chain.invoke("tell me about elon musk, his education, his companies, achievements. Gice me response in json format")

In [ ]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate(template= " You are an expert in {subject}. explain about {topic} in simple terms. Give the response in json format")
template.invoke( {
    "subject": "AI",
    "topic": "RAG"
})


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an expert in {subject}."),
        ("human" , "explain about {topic} in simple terms. Give the response in json format")
    ]
)

In [ ]:
from langchain_core.output_parsers import StrOutputParser

chain = template | llm_client | JsonOutputParser()

response= chain.invoke( {
    "subject": "AI",
    "topic": "RAG"
})
response


In [27]:
result = llm_client.invoke("tell me about elon musk" \
" Give me response in json format")

result.content

'{\n  "name": "Elon Musk",\n  "summary": "Elon Musk is a technology entrepreneur, investor, and CEO known for founding and leading companies in electric vehicles, space exploration, and artificial intelligence.",\n  "known_for": [\n    "Tesla",\n    "SpaceX",\n    "X (formerly Twitter)",\n    "Neuralink",\n    "The Boring Company",\n    "PayPal co-founding (via X.com)"\n  ],\n  "bio": {\n    "born": "June 28, 1971",\n    "birthplace": "Pretoria, South Africa",\n    "nationality": "South African-born entrepreneur who later became a citizen of the United States",\n    "education": "Attended Queen\'s University and the University of Pennsylvania"\n  },\n  "notable_achievements": [\n    "Helped popularize electric vehicles through Tesla",\n    "Advanced private spaceflight with SpaceX",\n    "Led the first privately developed company to send astronauts to the International Space Station",\n    "Expanded consumer interest in reusable rockets and satellite internet"\n  ],\n  "controversies":

In [29]:
from pydantic import BaseModel, Field

class Person(BaseModel):
    first_name: str = Field(description="First name of the person")
    last_name: str =  Field(description="LAst name of the person")
    age: int = Field(description= "age of the person in days")

In [34]:
from langchain_core.output_parsers import PydanticOutputParser

parser = PydanticOutputParser(pydantic_object=Person)
parser.get_format_instructions()

'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"first_name": {"description": "First name of the person", "title": "First Name", "type": "string"}, "last_name": {"description": "LAst name of the person", "title": "Last Name", "type": "string"}, "age": {"description": "age of the person in days", "title": "Age", "type": "integer"}}, "required": ["first_name", "last_name", "age"]}\n```'

In [35]:
template = ChatPromptTemplate.from_messages(
    [
        ("system", " Answer the below query {format_instructions}"),
        ("human", "tell me about {person} ")
    ]
).partial(format_instructions = parser.get_format_instructions())

chain = template | llm_client | parser

response = chain.invoke({
    "person": "Narendra modi"
})
response

Person(first_name='Narendra', last_name='Modi', age=25200)